<h4>Pirate FM</h4>
<h5>Arrrr, here lies the code used to scrape the table from The Pirate Archive, storing it in a dataframe, cleaning up the dataframe and turning them into csv files. The git repository is found here (https://git.arts.ac.uk/23008862/Data-Visualisation) and the live website is found here (https://cci.arts.ac.uk/~elowen/PirateFM/)​</h5>


In [2]:
from bs4 import BeautifulSoup
import numpy as np
import requests
import re

In [3]:
url = "https://www.thepiratearchive.net/complete-london-stations-list/"
page = requests.get(url)
#BeautifulSoup parser
soup = BeautifulSoup(page.content, features="html.parser")


In [4]:
table = soup.find_all("table")
#can also do table = soup.find("table", class_="wikitable sortable")

In [5]:
#getting the header with the th tags
header = soup.find_all("th")
print(header)

[<th class="column-1">Station Name</th>, <th class="column-2">Frequency</th>, <th class="column-3">Dates On Air</th>]


In [6]:
header_titles = [title.text.strip() for title in header]
print(type(header_titles))

<class 'list'>


In [7]:
import pandas as pd

In [8]:
df = pd.DataFrame(columns = header_titles)
df

,Station Name,Frequency,Dates On Air


In [9]:
#th tags were titles, tr is rows, td is data, tr's encompass all the td's in a row
column_data = soup.find_all("tr")


In [10]:
for row in column_data[1:]:
    row_data = row.find_all("td")
    individual_row_data = [data.text.strip() for data in row_data]
    
    #looping through all the data and adding it to the df
    length = len(df)
    df.loc[length] = individual_row_data

df

,Station Name,Frequency,Dates On Air
0,2GFM,99.7,1999 - 2000
1,3BR - Three Boroughs Radio - N London,1125 & 1350kHz MW,1985 - 1986
2,9Nine3,99.3,
3,Abyss FM,96.5,1999 - July 2000
4,Activity FM,93.0,Nov 1982 - May 1983 (Sundays)
...,...,...,...
556,X-rated,100.7,
557,Xtreme - North London,101.6,2000 -
558,Y2K,90.6,2000 -
559,Zone FM,100.6 & 105.1 & 106.4,1992 - 1997


In [11]:
#creating series from Dates on air
year_column = df.filter(like="Dates On Air")
print(year_column)

                      Dates On Air
0                      1999 - 2000
1                      1985 - 1986
2                                 
3                 1999 - July 2000
4    Nov 1982 - May 1983 (Sundays)
..                             ...
556                               
557                         2000 -
558                         2000 -
559                    1992 - 1997
560                    1993 - 1994

[561 rows x 1 columns]


Here we face our first problem, the dates are difficult to filter through because they:
 - vary in length
 - some values include days and months
 - a lot of these values include a dash

Sometimes this cell has trouble running, if this happens try pressing the "Run All" button.

In [12]:
#to clean up years, if num items in list is more than 4 (index 3) than remove the rest
def year_clean(val):
    #if the value is a string and a digit and 4 digits long, keep the value
    if isinstance(val, str) and val.isdigit() and len(val) == 4:
        return val
    #create a string with the years
    val_str = str(val)
    #searching val_str for this regex combination r = raw string,\b is word boundary, (19\d{2}) matches any number from 1900 to 1999, (20\d{2}) matching from 2000, 2099, so it's finding any 4-digit year from 1900–2099 that is a standalone word.
    match = re.search(r"\b(19\d{2}|20\d{2})\b", val_str)
    if match:
        #return the val_str that matches the regex
        return match.group(0)

df["Dates On Air"] = df["Dates On Air"].apply(year_clean)
df


,Station Name,Frequency,Dates On Air
0,2GFM,99.7,1999
1,3BR - Three Boroughs Radio - N London,1125 & 1350kHz MW,1985
2,9Nine3,99.3,None
3,Abyss FM,96.5,1999
4,Activity FM,93.0,1982
...,...,...,...
556,X-rated,100.7,None
557,Xtreme - North London,101.6,2000
558,Y2K,90.6,2000
559,Zone FM,100.6 & 105.1 & 106.4,1992


Next I'll filter through frequencies. This should leave us with a standerdised collection of frequencies to add to the dataframe (expl. 100.3, 85.03, 140.06).

In [13]:
freq_column = df.filter(like="Frequency")
print(freq_column)

                 Frequency
0                     99.7
1        1125 & 1350kHz MW
2                     99.3
3                     96.5
4                     93.0
..                     ...
556                  100.7
557                  101.6
558                   90.6
559  100.6 & 105.1 & 106.4
560            96.1 & 96.6

[561 rows x 1 columns]


In [14]:
#we cant be too particular with the entry length rule with the frequency as this can vary from index 2 - 5
def is_valid_freq(s):
    #remove spaces
    s = s.replace(" ", "")
    #allow digits and dot
    allowed_chars = set("0123456789.")
    #return the characters if the entry has allowed_char characters and length is <= 5
    return all(ch in allowed_chars for ch in s) and len(s) <= 5 

In [15]:
#creating a mask with the function
mask = df["Frequency"].apply(is_valid_freq)

#adding the mask to the frequency
df.loc[mask, "Frequency"]
df

,Station Name,Frequency,Dates On Air
0,2GFM,99.7,1999
1,3BR - Three Boroughs Radio - N London,1125 & 1350kHz MW,1985
2,9Nine3,99.3,None
3,Abyss FM,96.5,1999
4,Activity FM,93.0,1982
...,...,...,...
556,X-rated,100.7,None
557,Xtreme - North London,101.6,2000
558,Y2K,90.6,2000
559,Zone FM,100.6 & 105.1 & 106.4,1992


In [16]:
#Getting rid of empty stings by filling them with null values
#This also gets rid of the null values
clean = df[df['Dates On Air'].str.strip().astype(bool)]
clean

,Station Name,Frequency,Dates On Air
0,2GFM,99.7,1999
1,3BR - Three Boroughs Radio - N London,1125 & 1350kHz MW,1985
3,Abyss FM,96.5,1999
4,Activity FM,93.0,1982
5,Addiction FM,95.2,1998
...,...,...,...
554,WNK,,1987
557,Xtreme - North London,101.6,2000
558,Y2K,90.6,2000
559,Zone FM,100.6 & 105.1 & 106.4,1992


<i>I'll do this to test how I'll be able to check which year to display in the sketch.js file:

1. Get the active_year value from the sketch.js
2. Split the active_year value into a list of characters exp ['1', '9', '9', '9']
3. Create a script that takes the active_year and finds any matches in the csv
4. Create a script that bundles years by frequency so its formatted like "22.3 - name.fm, name.fm, name.fm"
5. Make it return all the entries to sketch.js
6. Make a loop that creates a p for every frequency and year </i>


<h3>New Idea!</h3>
I have decided to scrap the preivous idea, instead I will be creating 10 new dataframes, one for each year, a much cleaner option.
To do this I must:
- Combine the dates on air strings back together
- If the dates on air value = 1990/1991/1992... create a dataframe for it
- create a csv with the 10 dataframes
sketch.js:
- preload all of the csv's, assign them all their own value
- if active_year == any of the years, display the years

But doing it this way would mean I'd probably have to create additional dataframes for the year in alphabetical and frequency order. I should make a folder with all 20 of these dataframes. 

In [17]:
clean["Dates On Air"] = clean["Dates On Air"].apply(lambda i: "".join(i))


C:\Users\Elune\AppData\Local\Temp\ipykernel_8496\1861070097.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean["Dates On Air"] = clean["Dates On Air"].apply(lambda i: "".join(i))


In [18]:
year = 1989
dfs_by_year = {}

while year < 2000:
    year += 1
    #converting this to string so i can use .contains()
    new_year = str(year)
    #creating a mask that only contains years from 1990 - 2000
    mask = clean["Dates On Air"].str.contains(new_year, na=False)
    dfs_by_year["df" + new_year] = clean[mask]


In [19]:
#I created this as a way to catagorise stations by alphabet, unfortunately I forgot that the dataframes were already in aphabetical order as seen above, I've kept this in for documentation

year = 1989
dfs_by_alphabet = {}

while year < 2000:
    year += 1
    newyear = str(year)

    df_year = dfs_by_year["df" + newyear]
    sorted_df = df_year.sort_values(by="Station Name")
    dfs_by_alphabet["df" + newyear] = sorted_df

dfs_by_alphabet["df1991"]
   

,Station Name,Frequency,Dates On Air
56,Chillin',97.6 & 97.8 & 102.9 & 104.4,1991
94,Destiny FM,99.3,1991
103,Dream Music Radio - East London,88.0,1991
112,Elite FM,98.3 & 104.7,1991
114,Elite Radio,98.0 & 98.5,1991
138,First Love Community Radio,101.8,1991
155,Frequency FM - North London,101.9,1991
175,Genesis Radio,91.6 & 91.8,1991
213,Impact FM,88.2 & 104.4,1991
249,Kool FM,94.6,1991


Now I'm going to sort these by frequency. In the data folder you will also see a file called "clean", this is the clean dataframe with all of the stations, this was used as a test dataset and was orignially situated in the overall PirateFm folder for ease of access.

In [20]:
year = 1989
dfs_by_frequency = {}

while year < 2000:
    year += 1
    newyear = str(year)

    df_year = dfs_by_year["df" + newyear]
    sorted_df = df_year.sort_values(by="Frequency")
    dfs_by_frequency["df" + newyear] = sorted_df

dfs_by_frequency["df1991"]
#converting to csv files 
#dfs_by_frequency["df1990"].to_csv("data/dfs_by_frequency_1990", sep=',', index=False, encoding='utf-8')
   

,Station Name,Frequency,Dates On Air
138,First Love Community Radio,101.8,1991
155,Frequency FM - North London,101.9,1991
508,Trance FM - Wandsworth,105.4,1991
103,Dream Music Radio - East London,88.0,1991
213,Impact FM,88.2 & 104.4,1991
463,Station FM,89.8,1991
175,Genesis Radio,91.6 & 91.8,1991
330,Powerjam,92.0,1991
547,Weekend Rush,92.3,1991
249,Kool FM,94.6,1991
